In [1]:
!pip install pandas==2.3.3
!pip install wfdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 106.7 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4

# DATA PREP

In [2]:
# ============================================================
# DATA PREP — PHYSIONET 2017 (DROP-IN REPLACEMENT)
# ============================================================

import os
import numpy as np
import pandas as pd
import wfdb
from collections import Counter
from scipy.signal import resample

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
np.random.seed(42)
tf.random.set_seed(42)

FS = 300
TARGET_FS = 300
SEG_LEN = 9000

ROOT_DIR = "/kaggle/input/physionet-af-dataset-modified/PhysioNet_AF_Dataset_modified/physionet.org/files/challenge-2017/1.0.0"
TRAIN_DIR = os.path.join(ROOT_DIR, "training")
VAL_DIR   = os.path.join(ROOT_DIR, "validation")
TRAIN_REF = os.path.join(TRAIN_DIR, "REFERENCE.csv")
VAL_REF   = os.path.join(VAL_DIR, "REFERENCE.csv")

def read_physionet_2017_records(base_dir, ref_csv):
    ref = pd.read_csv(ref_csv, header=None, names=["record", "label"])
    label_map = {"N": "N", "A": "A", "O": None, "~": None}

    Signals, Labels, Ids = [], [], []
    for _, row in ref.iterrows():
        rec = row["record"]
        lbl = label_map.get(row["label"])
        if lbl is None:
            continue
        try:
            record = wfdb.rdrecord(os.path.join(base_dir, rec))
            sig = record.p_signal[:, 0].astype(np.float32)
            sig = resample(sig, int(len(sig) * TARGET_FS / FS))
            if len(sig) >= SEG_LEN:
                Signals.append(sig)
                Labels.append(lbl)
                Ids.append(rec)
        except:
            continue
    return Signals, pd.Series(Labels, dtype="category"), np.array(Ids)

Signals_tr, Labels_tr, Ids_tr = read_physionet_2017_records(TRAIN_DIR, TRAIN_REF)
Signals_va, Labels_va, Ids_va = read_physionet_2017_records(VAL_DIR, VAL_REF)

Signals = Signals_tr + Signals_va
Labels  = pd.concat([Labels_tr, Labels_va], ignore_index=True)
Ids     = np.concatenate([Ids_tr, Ids_va])

records_df = pd.DataFrame({"id": Ids, "signal": Signals, "label": Labels})
print("Label distribution:", Counter(Labels))

train_ids, test_ids = train_test_split(
    records_df["id"].unique(), test_size=0.2, random_state=42
)

train_df = records_df[records_df["id"].isin(train_ids)].reset_index(drop=True)
test_df  = records_df[records_df["id"].isin(test_ids)].reset_index(drop=True)

def segment_signals_from_df(df, seg_len=SEG_LEN):
    X, Y = [], []
    for _, row in df.iterrows():
        x, y = row["signal"], row["label"]
        for i in range(len(x) // seg_len):
            X.append(x[i*seg_len:(i+1)*seg_len])
            Y.append(y)
    return X, Y

XTest, YTest = segment_signals_from_df(test_df)
le_test = LabelEncoder()
YTest_enc = le_test.fit_transform(YTest)

2026-02-02 07:46:38.331946: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770018398.523083      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770018398.581893      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770018399.060492      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770018399.060540      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770018399.060543      24 computation_placer.cc:177] computation placer alr

Label distribution: Counter({'N': 4686, 'A': 688})


# MODEL 2 — STFT + RESNET50 (REG + DROPOUT)

In [3]:
# ============================================================
# MODEL 2: STFT + RESNET50 — REG + DROPOUT
# ============================================================

import seaborn as sns
from scipy.signal import stft
from sklearn.metrics import roc_curve, roc_auc_score
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Conv2D, Flatten
from tensorflow.keras import mixed_precision
from sklearn.utils import resample as sk_resample
from tensorflow.keras.layers import (
    Input, Conv2D, Flatten, Dense,
    Dropout, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)

FS = 300
NFFT = 256
HOP = 32

def oversample(X, y):
    counter = Counter(y)
    max_count = max(counter.values())
    Xr, yr = [], []

    for cls in counter:
        Xc = [x for x, yy in zip(X, y) if yy == cls]
        yc = [cls] * len(Xc)

        Xu, yu = sk_resample(
            Xc, yc,
            replace=True,
            n_samples=max_count,
            random_state=42
        )

        Xr.extend(Xu)
        yr.extend(yu)

    return Xr, yr
    
def stft_image(x):
    f, _, Zxx = stft(x, fs=FS, nperseg=NFFT, noverlap=NFFT-HOP)
    S = np.log1p(np.abs(Zxx))
    return S.astype(np.float32)

def make_stft(X):
    return np.stack([stft_image(x) for x in X])[...,None]

tf.config.optimizer.set_jit(False)

Xtr, Ytr = segment_signals_from_df(train_df)
Xte, Yte = segment_signals_from_df(test_df)

Xtr, Ytr = oversample(Xtr, Ytr)

Xtr = make_stft(Xtr)
Xte = make_stft(Xte)

mean, std = Xtr.mean(), Xtr.std()+1e-8
Xtr = (Xtr-mean)/std
Xte = (Xte-mean)/std

le = LabelEncoder()
Ytr_enc = le.fit_transform(Ytr)
Yte_enc = le.transform(Yte)

inp = Input(shape=Xtr.shape[1:])
x = Conv2D(16, 3, padding="same", activation="relu")(inp)
x = BatchNormalization()(x)
base = ResNet50(include_top=False, weights=None, input_tensor=x)
x = Flatten()(base.output)
x = Dense(512, activation="relu", kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.3)(x)
out = Dense(2, activation="softmax", dtype="float32")(x)

model_stft = Model(inp,out)
model_stft.compile(optimizer=Adam(1e-3),
                   loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"])

model_stft.fit(Xtr, Ytr_enc, epochs=30, batch_size=16, verbose=0)

y_prob_stft = model_stft.predict(Xte)[:,1]
y_pred_stft = (y_prob_stft>=0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(Yte_enc, y_pred_stft).ravel()
print("\nSTFT RESULTS")
print("ACC :", accuracy_score(Yte_enc,y_pred_stft))
print("PREC:", precision_score(Yte_enc,y_pred_stft))
print("REC :", recall_score(Yte_enc,y_pred_stft))
print("SPEC:", tn/(tn+fp))
print("F1  :", f1_score(Yte_enc,y_pred_stft))
print("AUC :", roc_auc_score(Yte_enc, y_prob_stft))

np.save("y_prob_stft.npy", y_prob_stft)
np.save("stft_mean.npy", mean)
np.save("stft_std.npy", std)
# Save STFT ResNet50 model
model_stft.save("stft_resnet50_reg_dropout.h5")
print("STFT ResNet50 model saved.")

I0000 00:00:1770018492.509584      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1770018492.515548      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1770018527.240850      81 service.cc:152] XLA service 0x7e54280ad640 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1770018527.240886      81 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1770018527.240890      81 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1770018532.789423      81 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-02-02 07:48:59.754357: E external/local_xla/xla/stream_executor/cuda/c

38/38 ━━━━━━━━━━━━━━━━━━━━ 17s 229ms/step



STFT RESULTS
ACC : 0.9521008403361344
PREC: 0.9898063200815495
REC : 0.9538310412573674
SPEC: 0.9418604651162791
F1  : 0.9714857428714357
AUC : 0.9856821400831545
STFT ResNet50 model saved.
